# Chapter 4 lab — Reward design, and the robot that games it

Companion to **[Chapter 4 — Reinforcement Learning, Intuitively](https://kamatechorg.github.io/robo-greeno-data-a/tutorial/04-rl-intuition/)**.

Chapter 4's central claim: *the algorithm is a library import; the thing you actually engineer is the **reward function**, and the robot will exploit any loophole you leave.* We make that happen on purpose — train PPO on a naive reward, watch it cheat, then add the missing penalties and watch it behave.

**Turn on the GPU:** `Runtime → Change runtime type → T4 GPU`. ~8 minutes total.

## Step 1 — install the RL stack (~2 min)

In [ ]:
!pip install -q mujoco "gymnasium[mujoco]" stable-baselines3 mediapy
print("install OK")

## Step 2 — the three words, in code

We use Gymnasium's `Ant-v4` — the built-in env closest to a hexapod (a torso on multiple legs). Its **state** is joint angles + velocities, its **action** is the per-joint torques, and crucially it exposes the *pieces* of the reward so we can re-mix them.

In [ ]:
import gymnasium as gym, numpy as np
env = gym.make("Ant-v4")
s, info = env.reset(seed=0)
print("state  (what it senses): vector of", env.observation_space.shape[0], "numbers")
print("action (what it does):   vector of", env.action_space.shape[0], "torques")
env.step(env.action_space.sample())
print("reward pieces available:", [k for k in info if 'reward' in k or 'x_' in k])

## Step 3 — the naive reward: “just go forward”

We wrap the env so the reward is **only** forward velocity — no penalty for falling, flailing, or burning the servos. Chapter 4 predicts this learns to *dive forward and crash*: maximum velocity, briefly.

In [ ]:
class RewardWrapper(gym.Wrapper):
    """Recompute reward from scratch so we control every term."""
    def __init__(self, env, shaped):
        super().__init__(env); self.shaped = shaped
    def step(self, action):
        obs, _, term, trunc, info = self.env.step(action)
        fwd = info.get("x_velocity", 0.0)
        if self.shaped:
            # the 'negotiated treaty' from Chapter 4
            reward = (2.0*fwd
                      - 0.1*np.square(action).sum()      # don't burn servos
                      - 0.5*abs(obs[1])                  # keep torso level
                      - 5.0*float(term))                 # seriously, don't fall
        else:
            reward = fwd                                 # naive: speed only
        return obs, reward, term, trunc, info

def make(shaped):
    return RewardWrapper(gym.make("Ant-v4"), shaped)
print("wrapper ready")

## Step 4 — train both and compare (~5 min)

Short training (40k steps) — enough to show the *behaviour*, not to master walking. We track how often each policy **falls over** (episode terminates early) as the tell-tale sign of reward hacking.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

def train(shaped, steps=40_000):
    env = Monitor(make(shaped))
    model = PPO("MlpPolicy", env, verbose=0, device="cuda", seed=0)
    model.learn(total_timesteps=steps, progress_bar=True)
    # evaluate: average episode length (longer = stayed upright)
    lengths = []
    for _ in range(10):
        o, _ = env.reset(); done=False; n=0
        while not done:
            a, _ = model.predict(o, deterministic=True)
            o, _, term, trunc, _ = env.step(a); done = term or trunc; n += 1
        lengths.append(n)
    return model, np.mean(lengths)

print("training NAIVE reward (speed only)...")
naive_model, naive_len = train(shaped=False)
print("training SHAPED reward (the treaty)...")
shaped_model, shaped_len = train(shaped=True)

print(f"\n naive  reward: avg {naive_len:5.0f} steps before episode ended")
print(f" shaped reward: avg {shaped_len:5.0f} steps before episode ended")
print(" (longer = stayed upright instead of diving/crashing)")

## Step 5 — watch the naive policy cheat

In [ ]:
import os; os.environ.setdefault("MUJOCO_GL", "egl")
import mediapy as media

def rollout(model, label):
    env = gym.make("Ant-v4", render_mode="rgb_array")
    o, _ = env.reset(seed=1); frames=[]; done=False
    while not done and len(frames) < 300:
        a, _ = model.predict(o, deterministic=True)
        o, _, term, trunc, _ = env.step(a); done = term or trunc
        frames.append(env.render())
    print(f"{label}: episode lasted {len(frames)} frames")
    return frames

media.show_video(rollout(naive_model, "NAIVE (speed only)"), fps=30)
media.show_video(rollout(shaped_model, "SHAPED (treaty)"), fps=30)

!!! abstract "What you just saw"
    - **Same algorithm** (PPO, untouched) for both runs — only the reward changed.
    - The naive “speed only” policy lunges and ends its episodes fast: it found the loophole exactly as Chapter 4 warns.
    - Adding *stay-level*, *don't-burn-servos*, and *don't-fall* penalties — the “negotiated treaty” — keeps it upright longer.
    - Reward design, not the algorithm, was the whole job.

**Try it:** train 5× longer, or change the reward weights, and watch the behaviour shift. For the real hexapod + MJX + domain randomization, see the project's Week 6–8 phase. Back to the **[tutorial overview](https://kamatechorg.github.io/robo-greeno-data-a/tutorial/)**.